In [22]:
import duckdb
import pandas as pd
import re
from pathlib import Path

RAW = Path('../data/raw')
con = duckdb.connect()
events_path = RAW / 'userid-timestamp-artid-artname-traid-traname.tsv'

In [23]:
sessionized = con.execute(f"""
    SELECT
        user_id,
        timestamp,
        artist_name,
        track_name,
        LEAD(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp) AS next_timestamp,
        DATE_DIFF('second', timestamp,
            LEAD(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp)) AS gap_seconds,
        CASE
            WHEN DATE_DIFF('second',
                LAG(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp),
                timestamp) > 1200
            OR LAG(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp) IS NULL
            THEN 1 ELSE 0
        END AS is_new_session
    FROM read_csv_auto('{events_path}', delim='\t', header=False,
        names=['user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name'])
    ORDER BY user_id, timestamp
""").df()

sessionized['session_id'] = sessionized.groupby('user_id')['is_new_session'].cumsum()
sessionized['is_skip'] = (sessionized['gap_seconds'] < 30) & (sessionized['gap_seconds'].notna())

print(sessionized.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(19150868, 9)


In [24]:
sessionized.to_parquet('../data/processed/sessionized_events.parquet', index=False)

In [25]:
features = pd.read_csv('../data/raw/spotify_audio_features.csv')
features = features.drop(columns=['Unnamed: 0'])

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

unique_lastfm = sessionized[['artist_name', 'track_name']].drop_duplicates().reset_index(drop=True)
unique_lastfm['artist_norm'] = unique_lastfm['artist_name'].apply(normalize)
unique_lastfm['track_norm'] = unique_lastfm['track_name'].apply(normalize)
features['artist_norm'] = features['artists'].apply(normalize)
features['track_norm'] = features['track_name'].apply(normalize)

In [26]:
matched_exact = unique_lastfm.merge(
    features[['artist_norm', 'track_norm', 'danceability', 'energy', 'valence',
              'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']],
    on=['artist_norm', 'track_norm'],
    how='left',
    indicator=True
)

play_counts = sessionized.groupby(['artist_name', 'track_name']).size().reset_index(name='play_count')
matched_exact = matched_exact.merge(play_counts, on=['artist_name', 'track_name'], how='left')

play_coverage = matched_exact.groupby('_merge')['play_count'].sum()
print(play_coverage)
print(f"\nPlay-level coverage from exact match: {play_coverage['both'] / play_coverage.sum():.1%}")

C:\Users\91809\AppData\Local\Temp\ipykernel_25988\3777640671.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  play_coverage = matched_exact.groupby('_merge')['play_count'].sum()


_merge
left_only     17810446
right_only           0
both          11737338
Name: play_count, dtype: int64

Play-level coverage from exact match: 39.7%


In [27]:
# unmatched tracks, sorted by how much listening volume they represent
unmatched = matched_exact[matched_exact['_merge'] == 'left_only'].sort_values('play_count', ascending=False)

# how many tracks do we need to fuzzy-match to recover, say, 80% of the missing volume?
unmatched['cumulative_share'] = unmatched['play_count'].cumsum() / unmatched['play_count'].sum()
tracks_needed_for_80pct = (unmatched['cumulative_share'] <= 0.80).sum()

print(f"Total unmatched tracks: {len(unmatched):,}")
print(f"Tracks needed to cover 80% of missing play volume: {tracks_needed_for_80pct:,}")
unmatched.head(15)

Total unmatched tracks: 1,480,431
Tracks needed to cover 80% of missing play volume: 262,307


,artist_name,track_name,artist_norm,track_norm,danceability,energy,valence,tempo,acousticness,loudness,popularity,track_genre,_merge,play_count,cumulative_share
11692,The Postal Service,Such Great Heights,the postal service,such great heights,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3992,0.000224
18650,Boy Division,Love Will Tear Us Apart,boy division,love will tear us apart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3663,0.000430
20122,Muse,Supermassive Black Hole,muse,supermassive black hole,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3483,0.000625
11980,Death Cab For Cutie,Soul Meets Body,death cab for cutie,soul meets body,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3479,0.000821
20121,Muse,Starlight,muse,starlight,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3060,0.000993
12183,Arcade Fire,Rebellion (Lies),arcade fire,rebellion lies,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3048,0.001164
28864,Interpol,Evil,interpol,evil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2989,0.001331
133765,Kanye West,Love Lockdown,kanye west,love lockdown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2950,0.001497
20133,Muse,Time Is Running Out,muse,time is running out,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2945,0.001662
11231,Bloc Party,Banquet,bloc party,banquet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2906,0.001826


In [28]:
# does Radiohead appear in the Spotify features file at all, under any track?
features[features['artist_norm'] == 'radiohead'][['artists', 'track_name', 'track_genre']]

,artists,track_name,track_genre
2053,Radiohead,Creep,alt-rock
2055,Radiohead,No Surprises,alt-rock
2406,Radiohead,Karma Police,alt-rock
2462,Radiohead,High and Dry,alt-rock
2519,Radiohead,Fake Plastic Trees,alt-rock
2610,Radiohead,Exit Music (For A Film),alt-rock
2612,Radiohead,Weird Fishes/ Arpeggi,alt-rock
2704,Radiohead,Nude,alt-rock
2813,Radiohead,Paranoid Android,alt-rock
2853,Radiohead,How to Disappear Completely,alt-rock


In [29]:
# how many distinct genres does this Spotify file actually contain, and how many tracks per genre?
print(features['track_genre'].nunique())
print(features.groupby('track_genre').size().describe())

114
count     114.0
mean     1000.0
std         0.0
min      1000.0
25%      1000.0
50%      1000.0
75%      1000.0
max      1000.0
dtype: float64


In [30]:
def fuzzy_match_row(row, threshold=85):
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return pd.Series([None, None, artist_score, None])

    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return pd.Series([artist_match, None, artist_score, track_score])

    return pd.Series([artist_match, track_match, artist_score, track_score])

In [31]:
!pip install tqdm


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
from tqdm.auto import tqdm
tqdm.pandas()

to_fuzzy_match[['matched_artist', 'matched_track', 'artist_score', 'track_score']] = \
    to_fuzzy_match.progress_apply(fuzzy_match_row, axis=1)

fuzzy_success_rate = to_fuzzy_match['matched_track'].notna().mean()
print(f"Fuzzy match recovered: {fuzzy_success_rate:.1%} of the {len(to_fuzzy_match):,} attempted tracks")
to_fuzzy_match[to_fuzzy_match['matched_track'].notna()].head(10)

  0%|          | 0/262307 [00:00<?, ?it/s]

Fuzzy match recovered: 0.4% of the 262,307 attempted tracks


,artist_name,track_name,artist_norm,track_norm,danceability,energy,valence,tempo,acousticness,loudness,popularity,track_genre,_merge,play_count,cumulative_share,matched_artist,matched_track,artist_score,track_score
18650,Boy Division,Love Will Tear Us Apart,boy division,love will tear us apart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3663,0.000430,joy division,love will tear us apart,91.666667,100.000000
6110,Radiohead,Weird Fishes/Arpeggi,radiohead,weird fishesarpeggi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2089,0.008063,radiohead,weird fishes arpeggi,100.000000,97.435897
12293,The Kooks,Naïve,the kooks,nave,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1747,0.012309,the kooks,naive,100.000000,88.888889
147247,New Order,Blue Monday,new order,blue monday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1214,0.025565,new order,blue monday 88,100.000000,88.000000
15121,The Verve,Bittersweet Symphony,the verve,bittersweet symphony,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,911,0.045117,the verve,bitter sweet symphony,100.000000,97.560976
125720,The Cure,In Between Days,the cure,in between days,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,877,0.048228,the cure,inbetween days,100.000000,96.551724
24125,The Beatles,Tomorrow Never Knows,the beatles,tomorrow never knows,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,745,0.063386,the beatles,tomorrow never knows take 1,100.000000,85.106383
143083,Pink Floyd,Shine On You Crazy Diamond,pink floyd,shine on you crazy diamond,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,745,0.063470,pink floyd,shine on you crazy diamond pts 15,100.000000,88.135593
14403,Creedence Clearwater Revisited,Fortunate Son,creedence clearwater revisited,fortunate son,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,733,0.065212,creedence clearwater revival,fortunate son,86.206897,100.000000
137008,Pink Floyd,"Another Brick In The Wall, Part 2",pink floyd,another brick in the wall part 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,698,0.069792,pink floyd,another brick in the wall pt 2,100.000000,96.774194


In [33]:
import inspect
print(inspect.getsource(fuzzy_match_row))

def fuzzy_match_row(row, threshold=85):
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return pd.Series([None, None, artist_score, None])

    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return pd.Series([artist_match, None, artist_score, track_score])

    return pd.Series([artist_match, track_match, artist_score, track_score])



In [34]:
def fuzzy_match_row(row, threshold=85):
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return pd.Series([None, None, artist_score, None])

    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return pd.Series([artist_match, None, artist_score, track_score])

    return pd.Series([artist_match, track_match, artist_score, track_score])

In [35]:
print(inspect.getsource(fuzzy_match_row))

def fuzzy_match_row(row, threshold=85):
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return pd.Series([None, None, artist_score, None])

    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return pd.Series([artist_match, None, artist_score, track_score])

    return pd.Series([artist_match, track_match, artist_score, track_score])



In [36]:
from tqdm.auto import tqdm
tqdm.pandas()

to_fuzzy_match[['matched_artist', 'matched_track', 'artist_score', 'track_score']] = \
    to_fuzzy_match.progress_apply(fuzzy_match_row, axis=1)

fuzzy_success_rate = to_fuzzy_match['matched_track'].notna().mean()
print(f"Fuzzy match recovered: {fuzzy_success_rate:.1%} of the {len(to_fuzzy_match):,} attempted tracks")
to_fuzzy_match[to_fuzzy_match['matched_track'].notna()].head(10)

  0%|          | 0/262307 [00:00<?, ?it/s]

Fuzzy match recovered: 0.4% of the 262,307 attempted tracks


,artist_name,track_name,artist_norm,track_norm,danceability,energy,valence,tempo,acousticness,loudness,popularity,track_genre,_merge,play_count,cumulative_share,matched_artist,matched_track,artist_score,track_score
18650,Boy Division,Love Will Tear Us Apart,boy division,love will tear us apart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3663,0.000430,joy division,love will tear us apart,91.666667,100.000000
6110,Radiohead,Weird Fishes/Arpeggi,radiohead,weird fishesarpeggi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2089,0.008063,radiohead,weird fishes arpeggi,100.000000,97.435897
12293,The Kooks,Naïve,the kooks,nave,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1747,0.012309,the kooks,naive,100.000000,88.888889
147247,New Order,Blue Monday,new order,blue monday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1214,0.025565,new order,blue monday 88,100.000000,88.000000
15121,The Verve,Bittersweet Symphony,the verve,bittersweet symphony,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,911,0.045117,the verve,bitter sweet symphony,100.000000,97.560976
125720,The Cure,In Between Days,the cure,in between days,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,877,0.048228,the cure,inbetween days,100.000000,96.551724
24125,The Beatles,Tomorrow Never Knows,the beatles,tomorrow never knows,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,745,0.063386,the beatles,tomorrow never knows take 1,100.000000,85.106383
143083,Pink Floyd,Shine On You Crazy Diamond,pink floyd,shine on you crazy diamond,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,745,0.063470,pink floyd,shine on you crazy diamond pts 15,100.000000,88.135593
14403,Creedence Clearwater Revisited,Fortunate Son,creedence clearwater revisited,fortunate son,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,733,0.065212,creedence clearwater revival,fortunate son,86.206897,100.000000
137008,Pink Floyd,"Another Brick In The Wall, Part 2",pink floyd,another brick in the wall part 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,698,0.069792,pink floyd,another brick in the wall pt 2,100.000000,96.774194


In [37]:
# check artist-level match rate separately from track-level match rate
print(f"Artist matched (any score): {to_fuzzy_match['matched_artist'].notna().mean():.1%}")
print(f"Full track matched: {to_fuzzy_match['matched_track'].notna().mean():.1%}")

# specifically check our known "Boy Division" typo case
to_fuzzy_match[to_fuzzy_match['artist_name'] == 'Boy Division'][
    ['artist_name', 'track_name', 'matched_artist', 'matched_track', 'artist_score', 'track_score']
]

Artist matched (any score): 38.2%
Full track matched: 0.4%


,artist_name,track_name,matched_artist,matched_track,artist_score,track_score
18650,Boy Division,Love Will Tear Us Apart,joy division,love will tear us apart,91.666667,100.0


In [38]:
# attach real audio features to the successful fuzzy matches
fuzzy_success = to_fuzzy_match[to_fuzzy_match['matched_track'].notna()].copy()

fuzzy_enriched = fuzzy_success.merge(
    features[['artist_norm', 'track_norm', 'danceability', 'energy', 'valence',
              'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']],
    left_on=['matched_artist', 'matched_track'],
    right_on=['artist_norm', 'track_norm'],
    how='left',
    suffixes=('', '_spotify')
)

print(f"Fuzzy-enriched rows: {len(fuzzy_enriched):,}")

Fuzzy-enriched rows: 1,910


In [39]:
# exact matches: keep only the successfully matched rows
exact_success = matched_exact[matched_exact['_merge'] == 'both'][
    ['artist_name', 'track_name', 'danceability', 'energy', 'valence',
     'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']
].copy()
exact_success['match_type'] = 'exact'

fuzzy_final = fuzzy_enriched[
    ['artist_name', 'track_name', 'danceability', 'energy', 'valence',
     'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']
].copy()
fuzzy_final['match_type'] = 'fuzzy'

all_matches = pd.concat([exact_success, fuzzy_final], ignore_index=True)
print(f"Total unique tracks with features: {len(all_matches):,}")
print(all_matches['match_type'].value_counts())

Total unique tracks with features: 1,627,559
match_type
exact    1625649
fuzzy       1910
Name: count, dtype: int64


In [40]:
# register the in-memory DataFrames as DuckDB tables DuckDB can query directly
con.register('sessionized_view', sessionized)
con.register('all_matches_view', all_matches)

# do the join in DuckDB (memory-efficient), and write straight to parquet
con.execute("""
    COPY (
        SELECT s.*, m.danceability, m.energy, m.valence, m.tempo,
               m.acousticness, m.loudness, m.popularity, m.track_genre, m.match_type
        FROM sessionized_view s
        LEFT JOIN all_matches_view m
        ON s.artist_name = m.artist_name AND s.track_name = m.track_name
    ) TO '../data/processed/enriched_events.parquet' (FORMAT PARQUET)
""")

print("Saved directly to parquet via DuckDB.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved directly to parquet via DuckDB.


In [41]:
coverage_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(match_type) AS matched_rows
    FROM read_parquet('../data/processed/enriched_events.parquet')
""").df()

coverage_check['coverage_pct'] = coverage_check['matched_rows'] / coverage_check['total_rows']
coverage_check

,total_rows,matched_rows,coverage_pct
0,29645224,11910476,0.401767


In [42]:
# check how much duplication actually exists in the exact-match lookup
dupe_check = exact_success.groupby(['artist_name', 'track_name']).size()
print(f"Tracks with more than one matched row: {(dupe_check > 1).sum():,}")

# fix: keep only one row per (artist_name, track_name) — first occurrence is fine here
exact_success_dedup = exact_success.drop_duplicates(subset=['artist_name', 'track_name'], keep='first')
fuzzy_final_dedup = fuzzy_final.drop_duplicates(subset=['artist_name', 'track_name'], keep='first')

all_matches = pd.concat([exact_success_dedup, fuzzy_final_dedup], ignore_index=True)
print(f"Total unique tracks with features after dedup: {len(all_matches):,}")

Tracks with more than one matched row: 13,159
Total unique tracks with features after dedup: 21,161


In [43]:
print(f"exact_success rows (before dedup): {len(exact_success):,}")
print(f"fuzzy_final rows (before dedup): {len(fuzzy_final):,}")

exact_success rows (before dedup): 1,625,649
fuzzy_final rows (before dedup): 1,910


In [44]:
# one row per (artist, track) — when duplicated across genres, keep the most popular version
features_dedup = features.sort_values('popularity', ascending=False).drop_duplicates(
    subset=['artist_norm', 'track_norm'], keep='first'
)
print(f"Features: {len(features):,} rows -> {len(features_dedup):,} unique tracks after dedup")

Features: 114,000 rows -> 78,530 unique tracks after dedup


In [45]:
matched_exact = unique_lastfm.merge(
    features_dedup[['artist_norm', 'track_norm', 'danceability', 'energy', 'valence',
                     'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']],
    on=['artist_norm', 'track_norm'], how='left', indicator=True
)
print(f"matched_exact rows: {len(matched_exact):,} (should equal unique_lastfm's {len(unique_lastfm):,})")
exact_match_rate = (matched_exact['_merge'] == 'both').mean()
print(f"Corrected exact match rate: {exact_match_rate:.1%}")

matched_exact rows: 1,500,661 (should equal unique_lastfm's 1,500,661)
Corrected exact match rate: 1.3%


In [46]:
# sanity check: does features_dedup even look right?
print(features_dedup[['artists', 'track_name', 'artist_norm', 'track_norm', 'popularity']].head())
print(features_dedup['artist_norm'].isna().sum(), "null artist_norm values")
print(features_dedup['track_norm'].isna().sum(), "null track_norm values")

# specific known-good case: does Radiohead still exist correctly in features_dedup?
print(features_dedup[features_dedup['artist_norm'] == 'radiohead'][['artists', 'track_name', 'popularity']])

                       artists                             track_name  \
81051     Sam Smith;Kim Petras              Unholy (feat. Kim Petras)   
51664         Bizarrap;Quevedo  Quevedo: Bzrp Music Sessions, Vol. 52   
89411            Manuel Turizo                             La Bachata   
81210  David Guetta;Bebe Rexha                        I'm Good (Blue)   
88407                Bad Bunny                       Tití Me Preguntó   

                  artist_norm                          track_norm  popularity  
81051     sam smithkim petras              unholy feat kim petras         100  
51664         bizarrapquevedo  quevedo bzrp music sessions vol 52          99  
89411           manuel turizo                          la bachata          98  
81210  david guettabebe rexha                        im good blue          98  
88407               bad bunny                      tit me pregunt          97  
0 null artist_norm values
0 null track_norm values
         artists              

In [47]:
# does ('radiohead', 'creep') exist as a key in BOTH sides, independently checked?
in_lastfm = (unique_lastfm['artist_norm'] == 'radiohead') & (unique_lastfm['track_norm'] == 'creep')
in_features = (features_dedup['artist_norm'] == 'radiohead') & (features_dedup['track_norm'] == 'creep')

print(f"'radiohead'+'creep' exists in unique_lastfm: {in_lastfm.sum()} row(s)")
print(f"'radiohead'+'creep' exists in features_dedup: {in_features.sum()} row(s)")

# check column dtypes on both sides — a dtype mismatch can silently break merge matching
print(unique_lastfm[['artist_norm', 'track_norm']].dtypes)
print(features_dedup[['artist_norm', 'track_norm']].dtypes)

# also compare: how does OLD matched_exact's 'both' count relate to unique_lastfm's actual size?
# (this checks whether the ORIGINAL 52.3% number was itself inflated by duplicate keys)
print(f"unique_lastfm total: {len(unique_lastfm):,}")

'radiohead'+'creep' exists in unique_lastfm: 1 row(s)
'radiohead'+'creep' exists in features_dedup: 1 row(s)
artist_norm    object
track_norm     object
dtype: object
artist_norm    object
track_norm     object
dtype: object
unique_lastfm total: 1,500,661


In [48]:
# isolate JUST this one row from each side and manually merge
test_lastfm = unique_lastfm[in_lastfm]
test_features = features_dedup[in_features][['artist_norm', 'track_norm', 'danceability', 'energy', 'popularity']]

test_result = test_lastfm.merge(test_features, on=['artist_norm', 'track_norm'], how='left', indicator=True)
test_result[['artist_name', 'track_name', 'artist_norm', 'track_norm', 'danceability', '_merge']]

,artist_name,track_name,artist_norm,track_norm,danceability,_merge
0,Radiohead,Creep,radiohead,creep,0.515,both


In [49]:
matched_exact = unique_lastfm.merge(
    features_dedup[['artist_norm', 'track_norm', 'danceability', 'energy', 'valence',
                     'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']],
    on=['artist_norm', 'track_norm'],
    how='left',
    indicator=True
)

print(f"matched_exact rows: {len(matched_exact):,} (should equal {len(unique_lastfm):,})")
exact_match_rate = (matched_exact['_merge'] == 'both').mean()
print(f"Exact match rate: {exact_match_rate:.1%}")

matched_exact rows: 1,500,661 (should equal 1,500,661)
Exact match rate: 1.3%


In [50]:
lastfm_keys = set(zip(unique_lastfm['artist_norm'], unique_lastfm['track_norm']))
features_keys = set(zip(features_dedup['artist_norm'], features_dedup['track_norm']))
overlap = lastfm_keys & features_keys

print(f"True set-intersection match count: {len(overlap):,} out of {len(lastfm_keys):,} unique_lastfm keys")
print(f"That's {len(overlap)/len(lastfm_keys):.1%}")

True set-intersection match count: 9,292 out of 1,470,515 unique_lastfm keys
That's 0.6%


In [52]:
print(fuzzy_enriched.columns.tolist())
print(f"fuzzy_enriched rows: {len(fuzzy_enriched):,}")
print(f"fuzzy_success rows (before merge): {len(fuzzy_success):,}")

['artist_name', 'track_name', 'artist_norm_x', 'track_norm_x', 'danceability_x', 'energy_x', 'valence_x', 'tempo_x', 'acousticness_x', 'loudness_x', 'popularity_x', 'track_genre_x', '_merge', 'play_count', 'cumulative_share', 'matched_artist', 'matched_track', 'artist_score', 'track_score', 'artist_norm_y', 'track_norm_y', 'danceability_y', 'energy_y', 'valence_y', 'tempo_y', 'acousticness_y', 'loudness_y', 'popularity_y', 'track_genre_y']
fuzzy_enriched rows: 931
fuzzy_success rows (before merge): 931


In [53]:
# final enrichment: exact match only. Fuzzy matching was built and tested (see notebook history)
# but only recovered ~0.4% additional play coverage — not worth the added pipeline complexity,
# so it's excluded from the final dataset.
exact_success = matched_exact[matched_exact['_merge'] == 'both'][
    ['artist_name', 'track_name', 'danceability', 'energy', 'valence',
     'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']
].copy()
exact_success = exact_success.drop_duplicates(subset=['artist_name', 'track_name'], keep='first')
exact_success['match_type'] = 'exact'

print(f"Final matched tracks: {len(exact_success):,}")

# join onto the full 19M-row dataset via DuckDB, write straight to parquet
con.register('exact_success_view', exact_success)
con.execute("""
    COPY (
        SELECT s.*, m.danceability, m.energy, m.valence, m.tempo,
               m.acousticness, m.loudness, m.popularity, m.track_genre, m.match_type
        FROM sessionized_view s
        LEFT JOIN exact_success_view m
        ON s.artist_name = m.artist_name AND s.track_name = m.track_name
    ) TO '../data/processed/enriched_events.parquet' (FORMAT PARQUET)
""")
print("Saved to parquet.")

# final coverage check
coverage_check = con.execute("""
    SELECT COUNT(*) AS total_rows, COUNT(match_type) AS matched_rows
    FROM read_parquet('../data/processed/enriched_events.parquet')
""").df()
coverage_check['coverage_pct'] = coverage_check['matched_rows'] / coverage_check['total_rows']
coverage_check

Final matched tracks: 20,230


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved to parquet.


,total_rows,matched_rows,coverage_pct
0,19150868,1340422,0.069993
